# 04. Feature Engineering

하나의 통합 모델로 1~7일 뒤의 일별 판매량을 예측하기 위해, 기준일까지 확인 가능한 판매 이력과 각 목표일에 미리 알 수 있는 정보를 표 형태로 구성합니다. 전체 원본으로 과거 변수를 계산한 뒤 최근 365개 예측 기준일만 Horizon별로 확장하고 Parquet으로 저장합니다.

## 실행 환경


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))


## 패키지 준비

Seaborn과 나눔고딕 글꼴을 설치합니다.


In [ ]:
%pip install -q seaborn koreanize-matplotlib


## Library


In [ ]:
import pandas as pd

import seaborn as sns
import koreanize_matplotlib

from retail_forecast.config import load_config
from retail_forecast.features import get_feature_columns
from retail_forecast.pipeline import feature_cache_path, load_or_build_feature_table


## Feature 선정 근거

> **선정 원칙:** EDA에서 확인한 주간 반복·판매 규모 차이를 설명하고, 예측 기준일 `t`에 실제로 확보할 수 있는 값만 사용합니다. 목표일은 `d=t+h` (`h=1~7`)입니다.

| 구분 | 사용 Feature | 기준 날짜 | 채택 이유 |
|---|---|---|---|
| 최근 수요 | `sales`, `sales_lag_1/7/14/28/56` | `t`; `t−1`, `t−7`, `t−14`, `t−28`, `t−56` | 직전 판매 수준과 주간 반복을 함께 반영. 56일은 더 긴 주간 패턴을 확인하기 위한 후보 |
| 수요 안정화 | `sales_mean_7/14/28` | `t−1`까지의 최근 7·14·28개 관측값 | 하루 판매량의 일시적인 변동을 완화 |
| 동일 요일 | `sales_same_weekday_last_week`, `sales_same_weekday_4w_mean` | `d−7`, `d−14`, `d−21`, `d−28` | Baseline의 주간 패턴을 모델에도 제공. `h≤7`이라 기준일 이후 판매량을 사용하지 않음 |
| 예측 거리·달력 | `forecast_horizon`, 목표일의 요일·월·주차·주말·공휴일 | 목표일 `d` | 1~7일 거리와 미리 아는 날짜 효과를 구분 |
| 매장·상품군 | `store_nbr`, `family`, 도시·주·점포 유형·군집 | 기준일에 알려진 속성 | 매장·상품군별 판매 규모와 지역 차이를 반영 |
| 프로모션·거래량 | `onpromotion`, `target_onpromotion`, `current_transactions` | 기준일 `t`; 목표일 프로모션은 사전 계획값만 | 판촉과 현재 점포 수요 수준을 반영 |

**사용 조건:** `sales`와 기준일 거래량은 `t` 영업 종료 후 예측할 때만 사용할 수 있습니다. 목표일 프로모션은 `t`에 7일 앞 계획까지 확정된 경우에만 사용합니다. 실무에서 확보할 수 없다면 해당 열을 제외해야 합니다.

**제외:** `target_sales`는 정답이므로 입력에서 제외합니다. 유가는 해석 가능성과 실효성이 낮아 사용하지 않습니다. 전체 이력 GPU 실험에서는 사전에 알 수 없는 지진·돌발 이벤트 변수도 제외합니다. 이 표는 사전 설계 근거이며, 각 변수의 실제 기여는 검증 점수와 중요도 분석으로 확인합니다.


In [2]:
sns.set_theme(style="whitegrid", font="NanumGothic")

config = load_config(PROJECT_ROOT / "configs" / "baseline.toml")
features = load_or_build_feature_table(config)
feature_columns = get_feature_columns(features)
print(f"origin range: {features['date'].min().date()} ~ {features['date'].max().date()}")
print(f"rows: {len(features):,}")
print(f"cache: {feature_cache_path(config)}")

origin range: 2016-08-09 ~ 2017-08-08
rows: 4,517,370
cache: C:\Users\PCuser\Desktop\retail-demand-forecasting\data\processed\features_h1-7_d365_l1-7-14-28-56_r7-14-28.parquet


In [3]:
feature_summary = pd.DataFrame({
    "dtype": features[feature_columns].dtypes.astype(str),
    "missing": features[feature_columns].isna().sum(),
    "unique": features[feature_columns].nunique(),
})
feature_summary

,dtype,missing,unique
store_nbr,int16,0,54
family,category,0,33
sales,float32,0,118950
onpromotion,int16,0,265
sales_lag_1,float32,0,118937
sales_lag_7,float32,0,118854
sales_lag_14,float32,0,118768
sales_lag_28,float32,0,118623
sales_lag_56,float32,0,118230
sales_mean_7,float32,0,157569


## 생성 결과 예시

In [4]:
example_columns = [
    "date", "target_date", "forecast_horizon", "store_nbr", "family",
    "sales", "sales_lag_7", "sales_lag_28", "sales_mean_7",
    "sales_same_weekday_last_week", "sales_same_weekday_4w_mean",
    "target_onpromotion", "target_dayofweek", "target_sales",
]
features[example_columns].sample(10, random_state=42).sort_values(["store_nbr", "family", "target_date"])

,date,target_date,forecast_horizon,store_nbr,family,sales,sales_lag_7,sales_lag_28,sales_mean_7,sales_same_weekday_last_week,sales_same_weekday_4w_mean,target_onpromotion,target_dayofweek,target_sales
1966107,2016-11-02,2016-11-06,4,3,LADIESWEAR,48.000000,38.000000,29.000000,44.285713,86.000000,102.750000,0.0,6,90.000000
3447310,2017-07-24,2017-07-30,6,19,HOME CARE,173.000000,184.000000,203.000000,176.285721,195.000000,224.000000,7.0,6,225.000000
2864658,2017-01-11,2017-01-16,5,24,PET SUPPLIES,18.000000,22.000000,16.000000,19.285715,22.000000,18.000000,0.0,0,15.000000
1641859,2017-02-17,2017-02-20,3,30,HARDWARE,1.000000,0.000000,0.000000,0.714286,0.000000,0.500000,0.0,0,0.000000
1054293,2017-01-08,2017-01-10,2,35,DAIRY,193.000000,0.000000,161.000000,205.285721,266.000000,197.750000,11.0,1,214.000000
1145878,2017-01-07,2017-01-09,2,42,PRODUCE,3991.642090,3628.770020,2256.125000,3016.091309,4256.300781,2954.404053,7.0,0,2307.433105
1167724,2017-05-13,2017-05-15,2,44,MEATS,1413.104980,1166.208008,915.718994,988.958435,721.487000,889.037292,0.0,0,729.127014
3759230,2017-03-22,2017-03-28,6,45,"LIQUOR,WINE,BEER",152.000000,154.000000,213.000000,203.571426,102.000000,120.250000,7.0,1,193.000000
1828722,2017-04-29,2017-05-02,3,46,BEAUTY,25.000000,13.000000,24.000000,10.571428,7.000000,11.750000,0.0,1,11.000000
3175935,2016-11-26,2016-12-01,5,50,POULTRY,527.268982,510.937012,446.414001,460.959137,413.510010,394.629517,1.0,3,363.618988


## 데이터 누수 점검

- `target_sales`는 X 변수에서 제외합니다.
- 판매 lag와 이동평균은 기준일 또는 그 이전 값으로만 계산합니다.
- 목표일 정보는 사전에 정해지는 프로모션과 달력 정보만 사용합니다.
- 거래량은 목표일 값이 아니라 예측 기준일의 관측값을 사용합니다.
- 유가는 일별 수요와의 관계를 명확히 설명하기 어려워 제외합니다.
- 검증 시작일 이후에 관측되는 정답은 학습에서 제외합니다.

In [5]:
leakage_checks = pd.Series({
    "target_is_excluded": "target_sales" not in feature_columns,
    "horizon_matches_dates": features["target_date"].sub(features["date"]).dt.days.eq(features["forecast_horizon"]).all(),
    "horizons_are_one_to_seven": set(features["forecast_horizon"].unique()) == set(range(1, 8)),
    "weekly_reference_is_known": features["seasonal_reference_date"].le(features["date"]).all(),
    "identifier_is_excluded": "id" not in feature_columns,
    "target_is_non_negative": features["target_sales"].ge(0).all(),
})
leakage_checks

target_is_excluded           True
horizon_matches_dates        True
horizons_are_one_to_seven    True
weekly_reference_is_known    True
identifier_is_excluded       True
target_is_non_negative       True
dtype: bool

`sales`는 라벨이 아니라 예측 기준일에 관측한 판매량이고, `target_sales`가 각 `forecast_horizon`의 실제 라벨입니다. 동일한 기준일이 1~7일의 행으로 확장되며, 모델은 과거 판매 변수와 목표일의 사전 확정 정보를 함께 학습합니다.